# Sales Data — Analysis and KPI Calculation

This notebook transforms the cleaned order-line data into financial fields, business KPIs, and analysis tables by product, category, city, month, and payment method.

## 1. Import Libraries and Define Paths

In [1]:
from math import isclose
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEAN_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cleaned_sales_data.xlsx"
)
REPORT_PATH = PROJECT_ROOT / "reports" / "sales_report.xlsx"

print("Clean data:", CLEAN_DATA_PATH)
print("Report output:", REPORT_PATH)
print("Clean dataset exists:", CLEAN_DATA_PATH.exists())

Clean data: c:\Users\Victus 16\sales-data-analysis\data\processed\cleaned_sales_data.xlsx
Report output: c:\Users\Victus 16\sales-data-analysis\reports\sales_report.xlsx
Clean dataset exists: True


## 2. Load and Validate the Cleaned Dataset

In [3]:
df = pd.read_excel(
    CLEAN_DATA_PATH,
    sheet_name="Cleaned_Data",
)

df["Order_Date"] = pd.to_datetime(df["Order_Date"])

assert df.duplicated().sum() == 0
assert df.isna().sum().sum() == 0
assert (df["Quantity"] > 0).all()
assert (df["Unit_Price_USD"] > 0).all()
assert df["Discount_Rate"].between(0, 1).all()

print("Clean dataset validation passed.")
print("Rows:", len(df))
print("Unique orders:", df["Order_ID"].nunique())

Clean dataset validation passed.
Rows: 1275
Unique orders: 797


## 3. Create Financial and Time Columns

Currency values are rounded to two decimal places at order-line level before aggregation. This matches normal invoice/accounting behavior.

- `Gross_Sales_USD = Quantity × Unit_Price_USD`
- `Discount_Amount_USD = Gross_Sales_USD × Discount_Rate`
- `Net_Revenue_USD = Gross_Sales_USD - Discount_Amount_USD`

In [4]:
df["Gross_Sales_USD"] = (
    df["Quantity"] * df["Unit_Price_USD"]
).round(2)

df["Discount_Amount_USD"] = (
    df["Gross_Sales_USD"] * df["Discount_Rate"]
).round(2)

df["Net_Revenue_USD"] = (
    df["Gross_Sales_USD"]
    - df["Discount_Amount_USD"]
).round(2)

df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.to_period("M").astype(str)
df["Month_Name"] = df["Order_Date"].dt.month_name()

df.head()

,Order_Line_ID,Order_ID,Order_Date,Customer_ID,Product,Category,City,Quantity,Unit_Price_USD,Discount_Rate,Payment_Method,Gross_Sales_USD,Discount_Amount_USD,Net_Revenue_USD,Year,Month,Month_Name
0,LINE-00001,ORD-2025-0001,2025-11-24,CUS-0058,Keyboard,Accessories,New York,3,77.12,0.15,PayPal,231.36,34.70,196.66,2025,2025-11,November
1,LINE-00002,ORD-2025-0002,2025-02-14,CUS-0303,Wireless Mouse,Accessories,Seattle,1,45.73,0.05,Credit Card,45.73,2.29,43.44,2025,2025-02,February
2,LINE-00003,ORD-2025-0003,2025-11-29,CUS-0280,USB-C Hub,Accessories,Unknown,3,59.87,0.10,Debit Card,179.61,17.96,161.65,2025,2025-11,November
3,LINE-00004,ORD-2025-0004,2025-12-24,CUS-0217,Webcam,Accessories,Austin,1,88.27,0.00,PayPal,88.27,0.00,88.27,2025,2025-12,December
4,LINE-00005,ORD-2025-0005,2025-06-26,CUS-0310,External SSD,Storage,Austin,5,147.08,0.05,Credit Card,735.40,36.77,698.63,2025,2025-06,June


## 4. Calculate Core KPIs

Average Order Value uses unique orders, not DataFrame rows. Average Selling Price is the net revenue per unit after discounts.

In [5]:
total_gross_sales = float(df["Gross_Sales_USD"].sum())
total_discount = float(df["Discount_Amount_USD"].sum())
total_revenue = float(df["Net_Revenue_USD"].sum())
total_orders = int(df["Order_ID"].nunique())
total_units = int(df["Quantity"].sum())

average_order_value = total_revenue / total_orders
average_selling_price = total_revenue / total_units
effective_discount_rate = total_discount / total_gross_sales

print(f"Gross sales: ${total_gross_sales:,.2f}")
print(f"Total discount: ${total_discount:,.2f}")
print(f"Net revenue: ${total_revenue:,.2f}")
print(f"Unique orders: {total_orders:,}")
print(f"Units sold: {total_units:,}")
print(f"Average order value: ${average_order_value:,.2f}")
print(f"Average selling price: ${average_selling_price:,.2f}")
print(f"Effective discount rate: {effective_discount_rate:.2%}")

Gross sales: $453,359.22
Total discount: $23,867.05
Net revenue: $429,492.17
Unique orders: 797
Units sold: 2,703
Average order value: $538.89
Average selling price: $158.89
Effective discount rate: 5.26%


## 5. Reusable Grouped Analysis Function

The function below avoids repeating the same `groupby` and aggregation logic for every business dimension.

In [6]:
def build_grouped_analysis(
    data: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    result = (
        data.groupby(group_columns, as_index=False)
        .agg(
            Units_Sold=("Quantity", "sum"),
            Orders=("Order_ID", "nunique"),
            Gross_Sales_USD=("Gross_Sales_USD", "sum"),
            Discount_Amount_USD=("Discount_Amount_USD", "sum"),
            Net_Revenue_USD=("Net_Revenue_USD", "sum"),
        )
    )

    money_columns = [
        "Gross_Sales_USD",
        "Discount_Amount_USD",
        "Net_Revenue_USD",
    ]
    result[money_columns] = result[money_columns].round(2)

    result["Average_Order_Value_USD"] = (
        result["Net_Revenue_USD"] / result["Orders"]
    ).round(2)

    result["Revenue_Share_Pct"] = (
        result["Net_Revenue_USD"] / total_revenue * 100
    ).round(2)

    return result.sort_values(
        "Net_Revenue_USD",
        ascending=False,
    ).reset_index(drop=True)

## 6. Product Analysis

In [7]:
product_analysis = build_grouped_analysis(
    df,
    ["Product", "Category"],
)

top_selling_product = (
    product_analysis
    .sort_values(
        ["Units_Sold", "Net_Revenue_USD"],
        ascending=[False, False],
    )
    .iloc[0]
)
top_revenue_product = product_analysis.iloc[0]

product_analysis

,Product,Category,Units_Sold,Orders,Gross_Sales_USD,Discount_Amount_USD,Net_Revenue_USD,Average_Order_Value_USD,Revenue_Share_Pct
0,Laptop,Computers,175,87,167212.44,9074.46,158137.98,1817.68,36.82
1,Monitor,Computers,223,111,62488.10,3047.14,59440.96,535.50,13.84
2,Office Chair,Furniture,221,103,52958.38,2823.49,50134.89,486.75,11.67
3,External SSD,Storage,226,108,34152.22,1698.06,32454.16,300.50,7.56
4,Headphones,Audio,223,108,28950.04,1433.27,27516.77,254.78,6.41
5,Bluetooth Speaker,Audio,241,102,26496.27,1369.56,25126.71,246.34,5.85
6,Webcam,Accessories,210,101,18900.42,1160.58,17739.84,175.64,4.13
7,Keyboard,Accessories,236,120,17678.73,794.77,16883.96,140.70,3.93
8,Desk Lamp,Furniture,275,129,14961.15,871.16,14089.99,109.22,3.28
9,USB-C Hub,Accessories,177,87,11417.28,643.66,10773.62,123.83,2.51


## 7. Category Analysis

In [8]:
category_analysis = build_grouped_analysis(
    df,
    ["Category"],
)

top_category = category_analysis.iloc[0]
category_analysis

,Category,Units_Sold,Orders,Gross_Sales_USD,Discount_Amount_USD,Net_Revenue_USD,Average_Order_Value_USD,Revenue_Share_Pct
0,Computers,398,184,229700.54,12121.60,217578.94,1182.49,50.66
1,Furniture,496,225,67919.53,3694.65,64224.88,285.44,14.95
2,Accessories,844,359,57932.42,3115.44,54816.98,152.69,12.76
3,Audio,464,203,55446.31,2802.83,52643.48,259.33,12.26
4,Storage,501,216,42360.42,2132.53,40227.89,186.24,9.37


## 8. City Analysis

`Unknown` remains in the table so revenue reconciles, but it is excluded when selecting the top named city.

In [9]:
city_analysis = build_grouped_analysis(
    df,
    ["City"],
)

known_city_analysis = city_analysis[
    city_analysis["City"] != "Unknown"
]
top_city = known_city_analysis.iloc[0]
city_analysis

,City,Units_Sold,Orders,Gross_Sales_USD,Discount_Amount_USD,Net_Revenue_USD,Average_Order_Value_USD,Revenue_Share_Pct
0,San Francisco,469,135,85441.65,4574.16,80867.49,599.02,18.83
1,Chicago,459,132,81766.90,4086.14,77680.76,588.49,18.09
2,Seattle,447,139,82035.10,5026.00,77009.10,554.02,17.93
3,Austin,483,148,73126.85,3332.56,69794.29,471.58,16.25
4,Boston,441,121,70136.85,3102.61,67034.24,554.00,15.61
5,New York,397,119,59539.96,3637.02,55902.94,469.77,13.02
6,Unknown,7,3,1311.91,108.56,1203.35,401.12,0.28


## 9. Monthly Analysis

In [10]:
monthly_analysis = build_grouped_analysis(
    df,
    ["Month"],
).sort_values("Month").reset_index(drop=True)

best_month = monthly_analysis.loc[
    monthly_analysis["Net_Revenue_USD"].idxmax()
]

monthly_analysis

,Month,Units_Sold,Orders,Gross_Sales_USD,Discount_Amount_USD,Net_Revenue_USD,Average_Order_Value_USD,Revenue_Share_Pct
0,2025-01,210,62,45869.34,2758.15,43111.19,695.34,10.04
1,2025-02,220,70,33181.60,1772.36,31409.24,448.70,7.31
2,2025-03,215,59,40323.32,2268.08,38055.24,645.00,8.86
3,2025-04,201,60,34002.16,1920.03,32082.13,534.70,7.47
4,2025-05,206,64,29107.87,1287.63,27820.24,434.69,6.48
5,2025-06,174,56,25255.96,1432.72,23823.24,425.42,5.55
6,2025-07,276,71,45004.50,1992.35,43012.15,605.80,10.01
7,2025-08,259,73,48455.95,2859.00,45596.95,624.62,10.62
8,2025-09,203,60,29686.86,2121.58,27565.28,459.42,6.42
9,2025-10,228,66,29359.75,1476.78,27882.97,422.47,6.49


## 10. Payment Method Analysis

In [11]:
payment_analysis = build_grouped_analysis(
    df,
    ["Payment_Method"],
)

top_payment_method = (
    payment_analysis
    .sort_values(
        ["Orders", "Net_Revenue_USD"],
        ascending=[False, False],
    )
    .iloc[0]
)

payment_analysis

,Payment_Method,Units_Sold,Orders,Gross_Sales_USD,Discount_Amount_USD,Net_Revenue_USD,Average_Order_Value_USD,Revenue_Share_Pct
0,Bank Transfer,658,204,127467.09,7166.57,120300.52,589.71,28.01
1,Debit Card,663,187,111182.69,5339.52,105843.17,566.01,24.64
2,Credit Card,678,198,108159.07,5558.00,102601.07,518.19,23.89
3,PayPal,695,204,104794.47,5625.29,99169.18,486.12,23.09
4,Unknown,9,4,1755.90,177.67,1578.23,394.56,0.37


## 11. Build the KPI Summary

In [12]:
kpi_summary = pd.DataFrame(
    [
        {"Metric": "Total Gross Sales (USD)", "Value": round(total_gross_sales, 2)},
        {"Metric": "Total Discount (USD)", "Value": round(total_discount, 2)},
        {"Metric": "Total Net Revenue (USD)", "Value": round(total_revenue, 2)},
        {"Metric": "Total Orders", "Value": total_orders},
        {"Metric": "Total Units Sold", "Value": total_units},
        {"Metric": "Average Order Value (USD)", "Value": round(average_order_value, 2)},
        {"Metric": "Average Selling Price (USD)", "Value": round(average_selling_price, 2)},
        {"Metric": "Effective Discount Rate (%)", "Value": round(effective_discount_rate * 100, 2)},
        {"Metric": "Top-Selling Product", "Value": top_selling_product["Product"]},
        {"Metric": "Highest-Revenue Product", "Value": top_revenue_product["Product"]},
        {"Metric": "Highest-Revenue Category", "Value": top_category["Category"]},
        {"Metric": "Highest-Revenue City", "Value": top_city["City"]},
        {"Metric": "Best Revenue Month", "Value": best_month["Month"]},
        {"Metric": "Most-Used Payment Method", "Value": top_payment_method["Payment_Method"]},
    ]
)

kpi_summary

,Metric,Value
0,Total Gross Sales (USD),453359.22
1,Total Discount (USD),23867.05
2,Total Net Revenue (USD),429492.17
3,Total Orders,797
4,Total Units Sold,2703
5,Average Order Value (USD),538.89
6,Average Selling Price (USD),158.89
7,Effective Discount Rate (%),5.26
8,Top-Selling Product,Desk Lamp
9,Highest-Revenue Product,Laptop


## 12. Reconcile Analysis Tables

Every grouped table must sum back to the same total revenue. This prevents silent omissions or double counting.

In [13]:
analysis_tables = {
    "Product": product_analysis,
    "Category": category_analysis,
    "City": city_analysis,
    "Month": monthly_analysis,
    "Payment Method": payment_analysis,
}

for analysis_name, analysis_table in analysis_tables.items():
    analysis_total = float(analysis_table["Net_Revenue_USD"].sum())

    assert isclose(
        analysis_total,
        total_revenue,
        abs_tol=0.01,
    )

    print(
        f"{analysis_name} analysis reconciled:",
        f"${analysis_total:,.2f}",
    )

Product analysis reconciled: $429,492.17
Category analysis reconciled: $429,492.17
City analysis reconciled: $429,492.17
Month analysis reconciled: $429,492.17
Payment Method analysis reconciled: $429,492.17


## 13. Export Analysis Tables to Excel

This workbook is the analytical foundation. Styling, charts, and dashboard layout will be added in the next project stage.

In [14]:
REPORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with pd.ExcelWriter(
    REPORT_PATH,
    engine="openpyxl",
) as writer:
    kpi_summary.to_excel(writer, sheet_name="Summary", index=False)
    df.to_excel(writer, sheet_name="Cleaned_Data", index=False)
    monthly_analysis.to_excel(writer, sheet_name="Monthly_Sales", index=False)
    product_analysis.to_excel(writer, sheet_name="Product_Analysis", index=False)
    category_analysis.to_excel(writer, sheet_name="Category_Analysis", index=False)
    city_analysis.to_excel(writer, sheet_name="City_Analysis", index=False)
    payment_analysis.to_excel(writer, sheet_name="Payment_Analysis", index=False)

print("Sales analysis workbook exported successfully.")
print("Output:", REPORT_PATH)

Sales analysis workbook exported successfully.
Output: c:\Users\Victus 16\sales-data-analysis\reports\sales_report.xlsx


## Key Results

- Gross sales: **$453,359.22**
- Total discount: **$23,867.05**
- Net revenue: **$429,492.17**
- Unique orders: **797**
- Units sold: **2,703**
- Average order value: **$538.89**
- Top-selling product by units: **Desk Lamp**
- Highest-revenue product: **Laptop**
- Highest-revenue category: **Computers**
- Highest-revenue named city: **San Francisco**
- Best revenue month: **2025-08**